# L09 · 合成数据录制与采数吞吐

本实验把 L08 的 scripted rollout 转换成一个小型持久化数据集：

```text
对齐 observation 与 action → 按 dataset FPS 采样 → 接受或丢弃整个 episode
→ 写入两条成功 episode → 重新打开并检查落盘数据
```

共享代码包负责 scene、scripted expert、recorder 和 LeRobot 集成；notebook 会直接展示 sampling decision、feature schema、success gate、writer lifecycle 和 readback check。

## 运行前准备

启动 Jupyter 前安装课程的数据依赖：

```bash
uv sync --locked --extra data
```

`ROBO_GENESIS_BACKEND=auto` 会在可用时选择已验证的 AMD backend，否则使用 CPU；设置为 `cpu` 可明确要求 CPU。`ROBO_GENESIS_RENDER` 默认为 `1`，因为两路 camera stream 都是本课数据。设为 `0` 只运行 sampling 和 schema 诊断，不会录制 dataset，也不能完成本课实验。修改任一变量后都应重启 kernel。

输出位置是 `ROBO_GENESIS_DATASETS_DIR/l09_banana_demo`。如果这个准确目录已经存在，notebook 会停止。只有确实要替换它时，才应在 kernel 启动前设置 `RG101_L09_OVERWRITE=1`。

先预测：

1. 从 100 Hz 降到 5 FPS 会保留哪些 control-step index？
2. 为什么从 100 Hz 降到 30 FPS 时，间隔需要在 3 step 和 4 step 之间变化？
3. 为什么一次失败 attempt 要整体丢弃？
4. 重新打开 dataset 后必须检查哪些内容？

In [ ]:
import importlib.metadata
import os
import shutil
from pathlib import Path

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.paths import DATASETS_DIR, OUTPUTS_DIR

lesson = load_course_manifest().lesson('L09')
assert lesson.slug == 'synthetic-data-recording-and-throughput'
assert lesson.duration_minutes == 120
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'planned'

backend_mode = os.environ.get('ROBO_GENESIS_BACKEND', 'auto').strip().lower()
if backend_mode not in {'auto', 'cpu'}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get('ROBO_GENESIS_RENDER', '1').strip()
if render_value not in {'0', '1'}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == '1'
overwrite_enabled = os.environ.get('RG101_L09_OVERWRITE', '0').strip() == '1'

REPO_ID = 'local/l09_banana_demo'
CONTROL_FPS = 100
DATASET_FPS = 5
IMG_WH = (160, 120)
TARGET_EPISODES = 2
MAX_ATTEMPTS = 10

dataset_parent = DATASETS_DIR.resolve()
dataset_root = (dataset_parent / 'l09_banana_demo').resolve()
if dataset_root.parent != dataset_parent or dataset_root.name != 'l09_banana_demo':
    raise ValueError(f'Unsafe dataset path: {dataset_root}')

cache_root = (OUTPUTS_DIR / 'l09_cache').resolve()
cache_paths = {
    'HF_DATASETS_CACHE': cache_root / 'huggingface_datasets',
    'XDG_CACHE_HOME': cache_root / 'xdg',
    'MPLCONFIGDIR': cache_root / 'matplotlib',
}
for variable, path in cache_paths.items():
    os.environ.setdefault(variable, str(path))
    Path(os.environ[variable]).mkdir(parents=True, exist_ok=True)
dataset_parent.mkdir(parents=True, exist_ok=True)

if render_enabled and dataset_root.exists():
    if not overwrite_enabled:
        raise FileExistsError(
            f'{dataset_root} already exists; choose another datasets directory or set '
            'RG101_L09_OVERWRITE=1 before restarting the kernel'
        )
    shutil.rmtree(dataset_root)

required_versions = {'genesis-world': '1.3.3', 'lerobot': '0.6.0', 'av': '15.1.0'}
installed_versions = {name: importlib.metadata.version(name) for name in required_versions}
version_mismatches = {
    name: (installed_versions[name], expected)
    for name, expected in required_versions.items()
    if installed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f'Dependency version mismatch: {version_mismatches}')

print(f'L09: {lesson.duration_minutes} min, hardware={lesson.hardware.value}, status={lesson.status.value}')
print(f'Requested backend={backend_mode}; render={render_enabled}; overwrite={overwrite_enabled}')
print(f'Dataset root: {dataset_root}')
if not render_enabled:
    print('DIAGNOSTIC MODE: no cameras, recording, persistence, readback, or visual evidence')

## 按 dataset rate 保留完整数据帧

每个 simulator step 之前都会调用 recorder，因此被保留的一行会把当前 measured state、当前两路 camera view 与即将执行的 command 配在一起。同一个 sampling decision 会保留或跳过这一整组数据。

![一次 step 前的采样决定会共同保留对齐的 observation 与 action；完整 attempt 随后再整体保存或丢弃。](../../docs/public/diagrams/l09-alignment-and-episode-transaction-zh.svg)

Sampling cell 会复现 `EpisodeRecorder` 的累加逻辑：先保留 callback 0，再以确定性方式分配无法整除的间隔。无需重新运行 Genesis，就可以把 `CANDIDATE_DATASET_FPS` 改为 10、20 或 30。运行前先预测间隔规律和保留帧数。

In [ ]:
import numpy as np

def describe_schedule(dataset_fps, total_control_steps=CONTROL_FPS):
    # Match EpisodeRecorder.reset(): preload one interval to retain callback 0.
    steps_per_frame = CONTROL_FPS / dataset_fps
    accum = steps_per_frame
    retained = []
    for control_step in range(total_control_steps):
        accum += 1.0
        if accum < steps_per_frame:
            continue
        accum -= steps_per_frame
        retained.append(control_step)
    gaps = np.diff(retained)
    logical_timestamps = np.arange(len(retained), dtype=float) / dataset_fps
    return {
        'indices': tuple(retained),
        'gaps': tuple(sorted(set(gaps.tolist()))),
        'average_fps': len(retained) / (total_control_steps / CONTROL_FPS),
        'logical_timestamps': logical_timestamps,
    }


five_fps_schedule = describe_schedule(DATASET_FPS)
thirty_fps_schedule = describe_schedule(30)

sampling_checks = {
    'five_fps_indices': five_fps_schedule['indices'] == (0, 19, 39, 59, 79, 99),
    'five_fps_gaps': five_fps_schedule['gaps'] == (19, 20),
    'five_fps_logical_time': np.allclose(
        five_fps_schedule['logical_timestamps'], np.arange(6) / 5
    ),
    'thirty_fps_count': len(thirty_fps_schedule['indices']) == 30,
    'thirty_fps_initial_indices': thirty_fps_schedule['indices'][:5] == (0, 3, 6, 10, 13),
    'thirty_fps_gaps': thirty_fps_schedule['gaps'] == (3, 4),
}
failed_sampling = [name for name, passed in sampling_checks.items() if not passed]
if failed_sampling:
    raise AssertionError('Sampling checks failed: ' + ', '.join(failed_sampling))

for rate, schedule in ((5, five_fps_schedule), (30, thirty_fps_schedule)):
    print(
        f'{CONTROL_FPS} Hz → {rate} FPS: count={len(schedule["indices"])}, '
        f'gaps={schedule["gaps"]}, average={schedule["average_fps"]:.1f}, '
        f'first indices={schedule["indices"][:8]}, '
        f'first logical times={schedule["logical_timestamps"][:5]}'
    )

CANDIDATE_DATASET_FPS = 30
candidate_schedule = describe_schedule(CANDIDATE_DATASET_FPS)
print(
    f'Candidate {CANDIDATE_DATASET_FPS} FPS retains {len(candidate_schedule["indices"])} '
    f'frames per simulated second; wall-clock throughput has not been measured.'
)

## 检查四项录制 feature

每个保留 frame 包含 Franka 的 measured joint position、专家给出的 9 维 position-target action，以及同步的 world 与 wrist RGB image。每帧还会传入自然语言 task；保存 episode 时，LeRobot 会自动添加 `index`、`episode_index`、`frame_index`、`timestamp` 和 `task_index` 等 bookkeeping field。

实际仿真夹爪用 force 闭合时，finger action entry 仍记录为 `0.0`。在这个数据集中，这两个零是供后续 policy 使用的“闭合”position-target proxy，并非 measured finger position 或 force command。

In [ ]:
from robo_genesis.record_dataset import JOINT_NAMES, build_features, task_description

USER_FEATURE_KEYS = {
    'observation.state',
    'action',
    'observation.images.world',
    'observation.images.wrist',
}
features = build_features(IMG_WH)
task_text = task_description('011_banana')
width, height = IMG_WH

feature_checks = {
    'four_user_features': set(features) == USER_FEATURE_KEYS,
    'nine_joint_names': len(JOINT_NAMES) == 9,
    'state_shape_and_names': tuple(features['observation.state']['shape']) == (9,)
    and tuple(features['observation.state']['names']) == tuple(JOINT_NAMES),
    'action_shape_and_names': tuple(features['action']['shape']) == (9,)
    and tuple(features['action']['names']) == tuple(JOINT_NAMES),
    'world_hwc_video': tuple(features['observation.images.world']['shape']) == (height, width, 3),
    'wrist_hwc_video': tuple(features['observation.images.wrist']['shape']) == (height, width, 3),
    'task_text': task_text == 'pick the banana and place it in the bowl',
}
failed_features = [name for name, passed in feature_checks.items() if not passed]
if failed_features:
    raise AssertionError('Feature checks failed: ' + ', '.join(failed_features))

for key, specification in features.items():
    print(f'{key}: dtype={specification["dtype"]}, shape={tuple(specification["shape"])}')
print('Joint order:', ', '.join(JOINT_NAMES))
print('Task:', task_text)

## 构建 recorder 与 writer

完整实验只初始化一次 Genesis，构建一个带有两台 camera 的 environment，并创建本地 LeRobot writer。这里关闭 domain randomization，让 L09 专注于录制机制；L11 会单独引入随机化。

禁用渲染时，这个 cell 不会创建上述任何 runtime object。该分支可以检查前面的纯逻辑，但不能代替 camera data。

In [ ]:
bundle = None
randomizer = None
recorder = None
dataset = None
actual_backend = 'not initialized'
runtime_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    import genesis as gs
    from lerobot.configs.video import RGBEncoderConfig
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

    from robo_genesis.build_scene import build_scene
    from robo_genesis.course_utils import select_backend
    from robo_genesis.randomize import EnvRandomizer, RandomizationConfig
    from robo_genesis.record_dataset import EpisodeRecorder

    backend = gs.cpu if backend_mode == 'cpu' else select_backend(prefer_rocm=True)
    gs.init(backend=backend, seed=0, precision='32', logging_level='warning')
    if getattr(gs, 'amdgpu', None) is not None and gs.backend == gs.amdgpu:
        actual_backend = 'amdgpu'
    elif gs.backend == gs.cpu:
        actual_backend = 'cpu'
    else:
        actual_backend = str(gs.backend)

    bundle = build_scene(
        show_viewer=False,
        n_envs=1,
        add_world_cam=True,
        add_wrist_cam=True,
        add_video_cam=False,
        draw_world_frame=False,
        scene_dr=None,
    )
    randomizer = EnvRandomizer(
        bundle,
        RandomizationConfig(randomize_pick=False, randomize_place=False, seed=0),
    )
    dataset = LeRobotDataset.create(
        repo_id=REPO_ID,
        root=dataset_root,
        fps=DATASET_FPS,
        features=features,
        robot_type='franka',
        use_videos=True,
        rgb_encoder=RGBEncoderConfig(vcodec='h264', video_backend='pyav'),
    )
    recorder = EpisodeRecorder(
        bundle,
        fps=DATASET_FPS,
        img_wh=IMG_WH,
        control_fps=CONTROL_FPS,
    )
    runtime_checks = {
        'supported_backend': actual_backend in {'cpu', 'amdgpu'},
        'explicit_cpu_honored': backend_mode != 'cpu' or actual_backend == 'cpu',
        'world_camera_ready': bundle.world_cam is not None,
        'wrist_camera_ready': bundle.wrist_cam is not None,
        'serial_scene': bundle.scene.n_envs in {0, 1},
        'writer_root': dataset.root.resolve() == dataset_root,
        'recorder_rate': recorder.fps == DATASET_FPS,
    }
    failed_runtime = [name for name, passed in runtime_checks.items() if not passed]
    if failed_runtime:
        raise AssertionError('Runtime setup checks failed: ' + ', '.join(failed_runtime))
    print(
        f'Runtime ready: backend={actual_backend} (requested={backend_mode}), '
        f'cameras=world+wrist, writer={dataset.root}'
    )
else:
    print('SKIP — rendering disabled; scene, cameras, recorder, and dataset writer were not created')

## 完成 attempt 后再决定是否保留

每次 attempt 都会重新设置 scene、recorder buffer 和 sampling phase。只有 release 和 settling 都结束后，`success` 才决定这次 transaction：成功且非空的 buffer 会被写入，并通过 `save_episode()` 结束当前 episode；失败 buffer 不会进入 writer。得到两条 accepted episode 或达到十次 attempt 后，再调用一次 `finalize()` 关闭整个 dataset。

In [ ]:
import time

from robo_genesis.grasp_demo import check_success, run_pick_place

attempt_summaries = []
accepted_frame_counts = []
attempts = 0
recording_wall_seconds = 0.0
dataset_finalized = False
recording_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    collection_started = time.perf_counter()
    try:
        while len(accepted_frame_counts) < TARGET_EPISODES and attempts < MAX_ATTEMPTS:
            episode_seed = attempts
            attempts += 1
            task = randomizer.reset(seed=episode_seed)
            if task.pick_object != '011_banana' or task.place_target != '024_bowl':
                raise AssertionError(f'Unexpected task: {task}')

            recorder.reset()
            success, _ = run_pick_place(bundle, task, recorder=recorder)
            confirmed_success = check_success(bundle, task)

            buffer_lengths = (
                len(recorder.states),
                len(recorder.actions),
                len(recorder.world_imgs),
                len(recorder.wrist_imgs),
            )
            states = np.stack(recorder.states)
            actions = np.stack(recorder.actions)
            world_images = np.stack(recorder.world_imgs)
            wrist_images = np.stack(recorder.wrist_imgs)
            attempt_checks = {
                'success_predicate_agrees': bool(success) == bool(confirmed_success),
                'equal_nonzero_buffers': len(set(buffer_lengths)) == 1 and buffer_lengths[0] > 0,
                'state_shape': states.shape == (len(recorder), 9),
                'action_shape': actions.shape == (len(recorder), 9),
                'state_action_floating': np.issubdtype(states.dtype, np.floating)
                and np.issubdtype(actions.dtype, np.floating),
                'state_action_finite': np.isfinite(states).all() and np.isfinite(actions).all(),
                'world_hwc_uint8': world_images.shape == (len(recorder), height, width, 3)
                and world_images.dtype == np.uint8,
                'wrist_hwc_uint8': wrist_images.shape == (len(recorder), height, width, 3)
                and wrist_images.dtype == np.uint8,
            }
            failed_attempt_checks = [
                name for name, passed in attempt_checks.items() if not passed
            ]
            if failed_attempt_checks:
                raise AssertionError(
                    f'Attempt {attempts} checks failed: ' + ', '.join(failed_attempt_checks)
                )

            decision = 'discarded'
            if success and len(recorder) > 0:
                recorder.flush_to(dataset, task_text)
                accepted_frame_counts.append(len(recorder))
                decision = 'accepted'

            attempt_summaries.append(
                {
                    'attempt': attempts,
                    'seed': episode_seed,
                    'success': bool(success),
                    'sampled_frames': len(recorder),
                    'decision': decision,
                }
            )
            summary = attempt_summaries[-1]
            print(
                f'Attempt {summary["attempt"]}: success={summary["success"]} → '
                f'{summary["decision"]}; dataset frames={summary["sampled_frames"]}'
            )
    finally:
        dataset.finalize()
        dataset_finalized = True
        recording_wall_seconds = time.perf_counter() - collection_started

    recording_checks = {
        'target_episode_count': len(accepted_frame_counts) == TARGET_EPISODES,
        'attempt_limit_honored': attempts <= MAX_ATTEMPTS,
        'positive_frame_counts': all(count > 0 for count in accepted_frame_counts),
        'dataset_finalized': dataset_finalized,
    }
    failed_recording = [name for name, passed in recording_checks.items() if not passed]
    if failed_recording:
        raise AssertionError('Recording checks failed: ' + ', '.join(failed_recording))
    print(
        f'Recorded {len(accepted_frame_counts)} accepted episodes in {attempts} attempts; '
        f'frames per accepted episode={accepted_frame_counts}'
    )
else:
    print('SKIP — no attempts were run and no dataset was written')

## 重新打开实际写入的内容

新建 metadata 和 reader object，可以检查持久化结果，而不是继续查看 writer 的内存状态。下面的检查会验证 episode boundary、logical timestamp、joint order、task text，以及两路 camera 的 PyAV 解码。

In [ ]:
metadata = None
recorded = None
readback_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    import torch
    from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

    metadata = LeRobotDatasetMetadata(REPO_ID, root=dataset_root)
    recorded = LeRobotDataset(REPO_ID, root=dataset_root, video_backend='pyav')
    rows = recorded.hf_dataset

    global_indices = np.asarray(rows['index'], dtype=np.int64)
    episode_indices = np.asarray(rows['episode_index'], dtype=np.int64)
    frame_indices = np.asarray(rows['frame_index'], dtype=np.int64)
    timestamps = np.asarray(rows['timestamp'], dtype=float)
    expected_total_frames = sum(accepted_frame_counts)
    episode_starts = [
        int(np.flatnonzero(episode_indices == episode)[0])
        for episode in range(TARGET_EPISODES)
    ]
    representative_samples = [recorded[index] for index in episode_starts]

    expected_features = USER_FEATURE_KEYS | {
        'index', 'episode_index', 'frame_index', 'timestamp', 'task_index'
    }
    episode_boundaries_ok = all(
        np.array_equal(
            frame_indices[episode_indices == episode],
            np.arange(np.sum(episode_indices == episode)),
        )
        and np.allclose(
            timestamps[episode_indices == episode],
            frame_indices[episode_indices == episode] / DATASET_FPS,
        )
        for episode in range(TARGET_EPISODES)
    )
    decoded_values_ok = all(
        sample['observation.state'].shape == (9,)
        and sample['action'].shape == (9,)
        and sample['observation.state'].dtype == torch.float32
        and sample['action'].dtype == torch.float32
        and torch.isfinite(sample['observation.state']).all().item()
        and torch.isfinite(sample['action']).all().item()
        and sample['observation.images.world'].shape == (3, height, width)
        and sample['observation.images.wrist'].shape == (3, height, width)
        and sample['observation.images.world'].dtype == torch.float32
        and sample['observation.images.wrist'].dtype == torch.float32
        and torch.isfinite(sample['observation.images.world']).all().item()
        and torch.isfinite(sample['observation.images.wrist']).all().item()
        and sample['task'] == task_text
        for sample in representative_samples
    )
    persistence_paths_ok = (dataset_root / 'meta' / 'info.json').is_file()
    persistence_paths_ok &= (dataset_root / 'meta' / 'tasks.parquet').is_file()
    persistence_paths_ok &= any((dataset_root / 'meta' / 'episodes').rglob('*.parquet'))
    persistence_paths_ok &= any((dataset_root / 'data').rglob('*.parquet'))
    persistence_paths_ok &= any((dataset_root / 'videos').rglob('*.mp4'))

    readback_checks = {
        'metadata_counts': metadata.fps == DATASET_FPS
        and metadata.total_episodes == TARGET_EPISODES
        and metadata.total_tasks == 1
        and metadata.total_frames == expected_total_frames > 0,
        'feature_set': set(metadata.features) == expected_features,
        'joint_names': tuple(metadata.features['observation.state']['names']) == tuple(JOINT_NAMES)
        and tuple(metadata.features['action']['names']) == tuple(JOINT_NAMES),
        'global_index_continuity': np.array_equal(
            global_indices, np.arange(expected_total_frames)
        ),
        'episode_index_monotonic': np.all(np.diff(episode_indices) >= 0),
        'episode_boundaries': episode_boundaries_ok,
        'decoded_values': decoded_values_ok,
        'persistent_files': persistence_paths_ok,
    }
    failed_readback = [name for name, passed in readback_checks.items() if not passed]
    if failed_readback:
        raise AssertionError('Readback checks failed: ' + ', '.join(failed_readback))
    print(
        f'Read back {metadata.total_frames} frames, {metadata.total_episodes} episodes, '
        f'{metadata.total_tasks} task; both camera streams decoded through PyAV.'
    )
else:
    print('SKIP — no persisted dataset exists to reopen in diagnostic mode')

## 检查边界、命令、状态与相机画面

下面的图全部来自重新打开的数据集。Timeline 应显示连续的 global index，以及在每条 episode 内重新开始的 frame/time。Arm 与 finger 使用不同 panel，因为它们的单位不同。Montage 从同一条持久化 episode 解码 start、middle 和 end，同一列的 world 与 wrist view 来自同一行数据。

In [ ]:
visual_checks = {'diagnostic_mode_requested': not render_enabled}

if render_enabled:
    import matplotlib.pyplot as plt

    state_rows = np.asarray(recorded.hf_dataset['observation.state'], dtype=float)
    action_rows = np.asarray(recorded.hf_dataset['action'], dtype=float)

    timeline, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].step(global_indices, episode_indices, where='post')
    axes[0].set_ylabel('episode_index')
    axes[0].set_title('Global rows continue while episode identity changes')
    axes[1].plot(global_indices, frame_indices, label='frame_index')
    axes[1].plot(global_indices, timestamps, label='timestamp (s)')
    axes[1].set_xlabel('global index')
    axes[1].set_title('Frame index and logical time restart for each episode')
    axes[1].legend()
    timeline.tight_layout()
    plt.show()

    arm_error = np.abs(action_rows[:, :7] - state_rows[:, :7])
    tracking_joint = int(np.argmax(np.max(arm_error, axis=0)))
    traces, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(action_rows[:, tracking_joint], '--', label='commanded target')
    axes[0].plot(state_rows[:, tracking_joint], label='measured qpos')
    axes[0].set_ylabel('arm angle (rad)')
    axes[0].set_title(f'Arm joint {tracking_joint}: command and measured state')
    axes[0].legend()
    for finger in range(2):
        axes[1].plot(action_rows[:, 7 + finger], '--', label=f'finger {finger} target')
        axes[1].plot(state_rows[:, 7 + finger], label=f'finger {finger} measured')
    axes[1].set_xlabel('global index')
    axes[1].set_ylabel('finger position (m)')
    axes[1].set_title('Finger proxy targets and measured positions')
    axes[1].legend(ncol=2)
    traces.tight_layout()
    plt.show()

    first_episode_rows = np.flatnonzero(episode_indices == 0)
    selected_rows = first_episode_rows[[0, len(first_episode_rows) // 2, -1]]
    selected_samples = [recorded[int(index)] for index in selected_rows]
    image_keys = ('observation.images.world', 'observation.images.wrist')
    decoded_images = {
        key: [sample[key].detach().cpu().numpy().transpose(1, 2, 0) for sample in selected_samples]
        for key in image_keys
    }
    visual_checks = {
        'state_action_rows': state_rows.shape == action_rows.shape == (metadata.total_frames, 9),
        'state_action_finite': np.isfinite(state_rows).all() and np.isfinite(action_rows).all(),
        'six_decoded_images': sum(len(images) for images in decoded_images.values()) == 6,
        'image_shapes': all(
            image.shape == (height, width, 3)
            for images in decoded_images.values()
            for image in images
        ),
        'nonempty_images': all(
            np.isfinite(image).all() and float(np.std(image)) > 0.0
            for images in decoded_images.values()
            for image in images
        ),
    }
    failed_visual = [name for name, passed in visual_checks.items() if not passed]
    if failed_visual:
        raise AssertionError('Visual checks failed: ' + ', '.join(failed_visual))

    montage, axes = plt.subplots(2, 3, figsize=(12, 6))
    for row, key in enumerate(image_keys):
        camera_name = key.rsplit('.', 1)[-1]
        for column, (dataset_index, sample, image) in enumerate(
            zip(selected_rows, selected_samples, decoded_images[key], strict=True)
        ):
            axes[row, column].imshow(np.clip(image, 0.0, 1.0))
            axes[row, column].set_title(
                f'{camera_name}: ep={int(sample["episode_index"])}, '
                f'frame={int(sample["frame_index"])}, t={float(sample["timestamp"]):.2f}s'
            )
            axes[row, column].axis('off')
    montage.suptitle('Start, middle, and end decoded from persisted episode 0')
    montage.tight_layout()
    plt.show()
else:
    print('SKIP — visual evidence requires the persisted camera dataset')

## 区分 dataset rate 与实测采数速度

完整运行时，最后一个 cell 会针对 attempt loop 与最终写盘过程，报告每真实秒写入的 frame 数和每真实小时接受的 episode 数。这些测量只描述本次运行及其配置；`DATASET_FPS=5` 并不保证程序每个 wall-clock second 都能写入 5 帧。Notebook 没有统计 reset 与 settling 中执行的每个 simulator step，因此不会报告 simulated-seconds ratio。

诊断模式只检查 timing 和 schema，因此不会使用完整实验的成功信息。

In [ ]:
final_checks = {
    'manifest_contract': lesson.status.value == 'planned'
    and lesson.duration_minutes == 120,
    'dependency_contract': not version_mismatches,
    'sampling_contract': all(sampling_checks.values()),
    'feature_contract': all(feature_checks.values()),
}

if render_enabled:
    final_checks.update(
        {
            'runtime_contract': all(runtime_checks.values()),
            'recording_contract': all(recording_checks.values()),
            'readback_contract': all(readback_checks.values()),
            'visual_contract': all(visual_checks.values()),
        }
    )
else:
    final_checks['diagnostic_boundary'] = (
        bundle is None and recorder is None and dataset is None and recorded is None
    )

failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError('L09 checks failed: ' + ', '.join(failed))

if render_enabled:
    dataset_bytes = sum(path.stat().st_size for path in dataset_root.rglob('*') if path.is_file())
    committed_frames_per_second = metadata.total_frames / recording_wall_seconds
    accepted_episodes_per_hour = len(accepted_frame_counts) * 3600.0 / recording_wall_seconds
    print(
        f'Observed throughput for attempt loop + finalize ({actual_backend}, n_envs=1, '
        f'{DATASET_FPS} FPS, {width}×{height}, h264; failed attempts included): '
        f'{committed_frames_per_second:.2f} committed frames/s, '
        f'{accepted_episodes_per_hour:.2f} accepted episodes/h'
    )
    print(
        f'Run evidence: attempts={attempts}, accepted_frames={accepted_frame_counts}, '
        f'elapsed={recording_wall_seconds:.2f}s, dataset_size={dataset_bytes / 1_000_000:.2f} MB'
    )
    print('L09 CHECK: PASSED')
else:
    print('L09 DIAGNOSTIC CHECK: PASSED — sampling and schema only')
    print('Core recording experiment: NOT COMPLETED (camera rendering is disabled)')

## 检查点与 L10 衔接

结合 sampling schedule、落盘 metadata、曲线和 montage，解释：

- 为什么记录 `(state_{t+1}, action_t)` 会产生一帧的 alignment error；
- 为什么从 100 Hz 得到 30 FPS 需要同时使用 3-step 与 4-step gap；
- 为什么 state、action 与两路 image 必须共用一个 sampling clock；
- 为什么 `save_episode()` 提交一条 accepted episode，而 `finalize()` 关闭整个 dataset；
- 为什么多个 simulator environment 本身不能保证 parallel recorder 正确或更快；
- 两条成功 episode 能证明什么，又不能说明 expert reliability、diversity 或 training quality 的哪些方面。

L10 将从这些持久化字段出发，解释 imitation-learning model 如何把 observation/action row 转换为训练 input 与 target。